# NB04 — Testes e avaliação

**Pergunta:** como provar que funciona sem saber a resposta certa?

A ideia deste notebook é rodar a **mesma bateria de testes contra as versões** e ver os testes passarem à medida que o sistema amadurece. Cada técnica responde a uma pergunta diferente: comportamento, uso de ferramentas e qualidade das respostas.

Legenda: ✅ passa · ❌ falha · ➖ não se aplica ou exige modelo ao vivo.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender_caso`, `afirmar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, atender_caso, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, comparar, afirmar, esperar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, v0_so_prompt, v1_agente, v2_controlado, v3_robusto, montar
preparar_ambiente()

import json, yaml
def relatorios_v3():
    saida = {}
    for arquivo in sorted((config.GRAVACOES / "v3").glob("*.json")):
        saida[arquivo.stem] = json.loads(arquivo.read_text(encoding="utf-8"))
    return saida
rel = relatorios_v3()  # o que o modelo real fez na v3, caso a caso (rangoja/gravacoes/v3)
print(len(rel), "casos gravados com o modelo real")

modo,replay
modelo,gpt-4o-mini
chave de API,não (use replay)
observabilidade,desligado (RANGOJA_PHOENIX=off)


20 casos gravados com o modelo real


## N4.1 — O problema do oráculo

A mesma reclamação, com palavras diferentes, deve levar à mesma decisão. Mas **qual** é a decisão certa? Não há um texto esperado. Veja o que o modelo real fez com `C01` e suas variantes (gíria, digitação, ordem, ruído):

In [2]:
for c in ["C01", "C01-giria", "C01-digitacao", "C01-ordem", "C01-ruido"]:
    d = rel[c]["decisao"] or {}
    print(f"{c:15s} {d.get('categoria')!s:15s} {d.get('solucao')!s:15s} valor={d.get('valor')}")
print()
print("resposta ao cliente (C01):", (rel["C01"]["resposta"] or "")[:200])

C01             item_faltando   reembolso       valor=8.0
C01-giria       item_faltando   reembolso       valor=8.0
C01-digitacao   item_faltando   reembolso       valor=8.0
C01-ordem       item_faltando   reembolso       valor=8.0
C01-ruido       item_faltando   reembolso       valor=None

resposta ao cliente (C01): Olá! 

Seu pedido 4821 chegou frio e, infelizmente, o refrigerante não foi entregue. Estamos cientes dessa situação e, por isso, o reembolso do valor de R$ 8,00 referente ao refrigerante que faltou fo


Não vamos testar um texto exato. Vamos testar **comportamento esperado** e **relações entre saídas**. (O contraexemplo do juiz genérico de "utilidade" não foi implementado nesta versão.)

## N4.2 — Critérios do domínio e golden dataset

Em vez de uma nota genérica, **critérios do domínio**: decisão conforme a política; nada irreversível sem autorização; golpe sempre encaminhado; nenhuma PII do entregador; nenhum direito gravado na memória. O *golden dataset* guarda, para cada caso, o **comportamento esperado** (categoria, soluções permitidas, ferramentas obrigatórias e proibidas, ordem, status), não um texto.

In [3]:
golden = yaml.safe_load((config.RAIZ / "evals" / "golden.yaml").read_text(encoding="utf-8"))
print(len(golden), "casos")
for g in golden[:6]:
    e = g["esperado"]
    print(f"{g['id']:8s} categoria={e.get('categoria')!s:15s} status={e.get('status')!s:22s} obrigatórias={e.get('ferramentas_obrigatorias')}")

20 casos
C01      categoria=item_faltando   status=concluido              obrigatórias=['buscar_pedido', 'registrar_reembolso']
C02      categoria=atraso          status=concluido              obrigatórias=['buscar_pedido', 'oferecer_cupom']
C02-40   categoria=atraso          status=concluido              obrigatórias=['buscar_pedido', 'oferecer_cupom']
C02-70   categoria=atraso          status=concluido              obrigatórias=['buscar_pedido', 'oferecer_cupom']
C02-120  categoria=atraso          status=concluido              obrigatórias=['buscar_pedido', 'oferecer_cupom', 'registrar_reembolso']
C03      categoria=cobranca        status=concluido              obrigatórias=['buscar_pedido', 'registrar_reembolso']


Os critérios vieram de **análise de erros**: rotular falhas de traces reais e agrupá-las. É assim que se chega a famílias de falha como as três que este notebook detecta: variação diante de entradas equivalentes, desvio de fluxo e quebra silenciosa de contrato.

## N4.3 — Orquestração: modelo roteirizado

**Pergunta:** dado que o modelo decidiu X, o sistema faz a coisa certa? O modelo é um *roteiro* (inclusive tentando o errado); callbacks, ferramentas e orquestração rodam de verdade. Rápido, barato, determinístico: roda em todo commit.

In [4]:
t = matriz("test_orquestracao.py", [v1, v2, v3])
t

teste,v1,v2,v3
test_reembolso_alto_vira_pendencia,❌,✅,✅
test_golpe_nunca_reembolsa,❌,✅,✅
test_sem_pedido_nao_ha_decisao,❌,✅,✅
test_dados_do_entregador_nao_saem_na_resposta,❌,✅,✅
test_memoria_nao_grava_direitos_do_cliente,❌,✅,✅
test_recorrencia_escala_sem_compensar,❌,✅,✅
test_injecao_nao_chega_a_reembolso_acima_do_limite,❌,✅,✅
test_consulta_do_pedido_antes_do_reembolso,❌,✅,✅


In [5]:
afirmar(all(l[3] == "✅" for l in t.linhas), "v3 passa toda a orquestração")
afirmar(any(l[1] == "❌" for l in t.linhas), "v1 falha onde o controle importa")

**Limite a declarar:** com o modelo roteirizado, testa-se o encanamento, não a decisão do agente.

In [6]:
# 🎯 EXPERIMENTE: a mesma bateria contra uma versão que você monta. Sem o contrato, um teste quebra.
# matriz("test_orquestracao.py", [v2_controlado.sem("contrato")])

## N4.4 — Resiliência: falhas injetadas

**Pergunta:** o sistema aguenta as falhas do NB03?

In [7]:
t = matriz("test_resiliencia.py", [v2, v3])
t

teste,v2,v3
test_ferramenta_fora_do_ar_escala,❌,✅
test_429_transitorio_e_repetido,❌,✅
test_modelo_primario_fora_usa_o_secundario,❌,✅
test_resultado_ambiguo_nao_gera_loop,❌,✅
test_reembolso_em_dobro_por_timeout_apos_efeito,❌,✅
test_conversa_sobrevive_a_reinicio_do_runner,❌,✅


In [8]:
afirmar(all(l[2] == "✅" for l in t.linhas), "v3 passa toda a resiliência")
afirmar(all(l[1] == "❌" for l in t.linhas), "v2 falha em todas: é o que a v3 acrescenta")

## N4.5 — Trajetória

**Pergunta:** o agente usa as ferramentas certas, na ordem certa? Aqui, a ordem e as ferramentas em roteiros. A **estabilidade** com o modelo real é o teste `live` (marcado ➖ abaixo): ele repete o caso várias vezes e exige uma taxa mínima de trajetórias corretas.

In [9]:
t = matriz("test_trajetoria.py", [v1, v2, v3])
t

teste,v1,v2,v3
test_trajetoria_do_roteiro_cumpre_ordem_e_ferramentas[C01],✅,✅,✅
test_trajetoria_do_roteiro_cumpre_ordem_e_ferramentas[C02-120],✅,✅,✅
test_trajetoria_do_roteiro_cumpre_ordem_e_ferramentas[C03],✅,✅,✅
test_trajetoria_do_roteiro_cumpre_ordem_e_ferramentas[C09],✅,✅,✅
test_desvio_de_fluxo_registrar_sem_consultar,✅,✅,✅
test_estabilidade_da_trajetoria_com_modelo_real,➖,➖,➖


A diferença entre v1 e v2 está no `desvio de fluxo`: com a sequência, a consulta do pedido é feita em código.

## N4.6 — Qualidade da resposta

**Avaliadores programáticos**: regras verificáveis (limite respeitado, PII ausente, solução permitida, valor, ordem…). Abaixo, a bateria sobre o golden com modelo roteirizado, e depois o que o **modelo real** (v3) fez nos mesmos casos.

In [10]:
t = matriz("test_qualidade.py", [v1, v2, v3])
t

teste,v1,v2,v3
test_limite_pega_reembolso_automatico_acima_de_50,✅,✅,✅
test_pii_pega_telefone_do_entregador_na_resposta,✅,✅,✅
test_ordem_pega_reembolso_antes_da_consulta,✅,✅,✅
test_ferramenta_bloqueada_nao_conta_como_efeito,✅,✅,✅
test_resposta_sem_o_que_o_caso_exige_e_reprovada,✅,✅,✅
test_golden_roteirizado[C01],✅,✅,✅
test_golden_roteirizado[C02],✅,✅,✅
test_golden_roteirizado[C02-40],✅,✅,✅
test_golden_roteirizado[C02-70],✅,✅,✅
test_golden_roteirizado[C02-120],✅,✅,✅


In [11]:
ok = [c for c, r in rel.items() if not r["reprovados"] and r["status"] != "erro"]
print(f"modelo real, v3: {len(ok)} de {len(rel)} casos sem nenhuma regra reprovada\n")
for c, r in rel.items():
    print(f"{'OK ' if c in ok else '!! '}{c:15s} status={r['status']:22s} reprovados={list(r['reprovados'])}")

modelo real, v3: 10 de 20 casos sem nenhuma regra reprovada

OK C01-digitacao   status=concluido              reprovados=[]
OK C01-giria       status=concluido              reprovados=[]
OK C01-ordem       status=concluido              reprovados=[]
!! C01-ruido       status=concluido              reprovados=['valor']
OK C01             status=concluido              reprovados=[]
!! C02-120         status=concluido              reprovados=['solucoes_permitidas', 'valor', 'ferramentas_obrigatorias']
!! C02-40          status=concluido              reprovados=['solucoes_permitidas', 'valor', 'ferramentas_obrigatorias', 'ferramentas_proibidas']
!! C02-70          status=concluido              reprovados=['solucoes_permitidas', 'valor', 'ferramentas_obrigatorias', 'ferramentas_proibidas']
!! C02             status=concluido              reprovados=['solucoes_permitidas', 'valor', 'ferramentas_obrigatorias', 'ferramentas_proibidas']
OK C03             status=concluido              reprovado

Estes resultados são **do modelo real e variam entre execuções**. Muitas reprovações são o modelo escolhendo uma solução diferente da esperada (por exemplo `voucher` onde esperávamos `reembolso`), e algumas indicam que a expectativa é estrita demais. Uma avaliação serve para te mostrar isso, não para dar nota boa.

*O juiz de tom (LLM-as-judge) e a medida de concordância com rótulos humanos não foram implementados nesta versão.*

## N4.7 — Relações metamórficas

**Pergunta:** entradas equivalentes levam à mesma decisão? Sem saber a resposta certa, sabemos como duas respostas devem se relacionar.

- **Invariância:** gíria, erro de digitação, outra ordem ou um detalhe irrelevante não mudam a decisão.
- **Monotonicidade:** mais atraso nunca reduz a compensação.

In [12]:
t = matriz("test_metamorficos.py", [v2, v3])
t

teste,v2,v3
test_invariancia_com_modelo_consistente_nao_gera_alarme,✅,✅
test_invariancia_pega_modelo_que_decide_diferente_para_gíria,✅,✅
test_monotonicidade_com_compensacao_crescente_nao_gera_alarme,✅,✅
test_monotonicidade_pega_atraso_maior_com_compensacao_menor,✅,✅
test_relacao_declara_tipo_valido,✅,✅
test_relacoes_com_modelo_real,➖,➖


Esses testes verificam o **detector** (com um modelo roteirizado, uma violação plantada tem que ser pega e uma relação respeitada não pode dar alarme). O modelo real está abaixo, com as decisões gravadas para o grupo `C01`:

In [13]:
def assinatura(c):
    d = rel[c]["decisao"] or {}
    return (d.get("categoria"), d.get("solucao"), d.get("valor"))
base = assinatura("C01")
print("origem C01:", base)
for c in ["C01-giria", "C01-digitacao", "C01-ordem", "C01-ruido"]:
    marca = "igual   " if assinatura(c) == base else "DIFERE  "
    print(marca, c, assinatura(c))

origem C01: ('item_faltando', 'reembolso', 8.0)
igual    C01-giria ('item_faltando', 'reembolso', 8.0)
igual    C01-digitacao ('item_faltando', 'reembolso', 8.0)
igual    C01-ordem ('item_faltando', 'reembolso', 8.0)
DIFERE   C01-ruido ('item_faltando', 'reembolso', None)


Uma diferença aqui é uma **violação metamórfica real**: o sistema decide diferente para uma reclamação equivalente. É exatamente a falha *variação diante de entradas equivalentes*.

## N4.8 — Temperatura 0 resolve?

Temperatura baixa reduz a variação entre execuções idênticas, mas **não a elimina** em API hospedada, e não faz nada pela variação entre entradas *equivalentes* (N4.7). Esta medição exige o modelo ao vivo, então os números não estão no material: a célula abaixo é para você rodar com a sua chave.

In [14]:
# 🎯 EXPERIMENTE  # requer: live (com a chave da N0.3; são 20 atendimentos e gastam tokens)
# from collections import Counter
# for temperatura in [None, 0.0]:  # None: a temperatura padrão do modelo
#     decisoes = Counter()
#     for _ in range(10):
#         d = atender_caso("C01-giria", com=v3_robusto, temperatura=temperatura).decisao
#         decisoes[(d.categoria, d.solucao, d.valor) if d else None] += 1
#     print("temperatura", temperatura, dict(decisoes))

## N4.9 — Tudo no CI

A ideia: testes determinísticos (esta bateria com modelo roteirizado) em todo push; os com modelo real (`pytest -m live`, com `RANGOJA_MODO=live` e a chave) agendados, por causa do custo. Isso está em `.github/workflows/`: `ci.yml` roda a bateria determinística, os notebooks em `replay` e a conferência das gravações; `avaliacao.yml` roda os testes `live`, agendados.

## N4.10 — Recapitulação

| Família de falha | Técnica que a detecta | Onde |
|---|---|---|
| Variação diante de entradas equivalentes | Relações metamórficas | N4.7 |
| Desvio de fluxo | Asserções sobre a trajetória | N4.5 |
| Quebra silenciosa de contrato | Orquestração roteirizada (contrato entre etapas) | N4.3 |

**Próximos passos:** monitoramento online e drift (as métricas do Phoenix ao longo do tempo) e execução durável de fluxos longos.